In [2]:
import django
import os

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'jury_soutenances.settings')
import django
import os

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'jury_soutenances.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"  # ← ajoute ça
django.setup()
django.setup()

In [3]:
from planification.models import Memoire, Enseignant, Disponibilite, Salle, Soutenance, Jury

print(Enseignant.objects.all())

<QuerySet [<Enseignant: Moussa Ouedraogo>, <Enseignant: Issouf Kabore>, <Enseignant: Dramane Sawadogo>, <Enseignant: Seydou Zongo>, <Enseignant: Aminata Traore>, <Enseignant: Ibrahim Compaore>, <Enseignant: Fatima Kindo>, <Enseignant: Oumar Tall>, <Enseignant: Mariam Diallo>, <Enseignant: Adama Barro>, <Enseignant: Pascal Some>, <Enseignant: Rasmane Toe>]>


In [4]:
from planification.models import Session, Enseignant, Etudiant, Salle, Memoire, Disponibilite
import datetime

# Créer une session
session = Session.objects.create(
    libelle="Session Juin 2026",
    date_debut=datetime.date(2026, 6, 20),
    date_fin=datetime.date(2026, 6, 25),
    statut='ouverte'
)

# Créer des enseignants
e1 = Enseignant.objects.create(nom="Kindo", prenom="Abdoul", email="kindo@esi.bf", grade="Professeur", specialite="Base de données")
e2 = Enseignant.objects.create(nom="Compaore", prenom="Jean", email="compaore@esi.bf", grade="Docteur", specialite="Algorithmique")
e3 = Enseignant.objects.create(nom="Tall", prenom="Ibrahim", email="tall@esi.bf", grade="Docteur", specialite="Sécurité")
e4 = Enseignant.objects.create(nom="Zongo", prenom="Ali", email="zongo@esi.bf", grade="Docteur", specialite="Algorithmique")
# Créer des étudiants
s1 = Etudiant.objects.create(matricule="ESI001", nom="Traore", prenom="Mamadou", filiere="Informatique", niveau="S4")
s2 = Etudiant.objects.create(matricule="ESI002", nom="Ouedraogo", prenom="Fatima", filiere="Informatique", niveau="S4")

# Créer des salles
salle1 = Salle.objects.create(nom="Salle A", capacite=30, batiment="Bloc 1")
salle2 = Salle.objects.create(nom="Salle B", capacite=20, batiment="Bloc 2")

# Créer des mémoires
m1 = Memoire.objects.create(etudiant=s1, session=session, titre="Gestion des stocks", type="Mémoire", specialite="Base de données")
m2 = Memoire.objects.create(etudiant=s2, session=session, titre="Algorithme de tri", type="Rapport", specialite="Algorithmique")

# Créer des disponibilités
Disponibilite.objects.create(enseignant=e1, session=session, jour=datetime.date(2026, 6, 20), heure_debut=datetime.time(8, 0), heure_fin=datetime.time(12, 0))
Disponibilite.objects.create(enseignant=e2, session=session, jour=datetime.date(2026, 6, 20), heure_debut=datetime.time(8, 0), heure_fin=datetime.time(12, 0))
Disponibilite.objects.create(enseignant=e3, session=session, jour=datetime.date(2026, 6, 20), heure_debut=datetime.time(14, 0), heure_fin=datetime.time(17, 0))
Disponibilite.objects.create(enseignant=e4, session=session, jour=datetime.date(2026, 6, 20), heure_debut=datetime.time(8, 0), heure_fin=datetime.time(12, 0))

print("Données créées ✅")

Données créées ✅


In [5]:
from planification.models import Enseignant, Disponibilite, Session
import datetime

session = Session.objects.first()
e4 = Enseignant.objects.create(nom="Zongo", prenom="Ali", email="zongo@esi.bf", grade="Docteur", specialite="Algorithmique")
Disponibilite.objects.create(
    enseignant=e4, 
    session=session, 
    jour=datetime.date(2026, 6, 20), 
    heure_debut=datetime.time(8, 0), 
    heure_fin=datetime.time(12, 0)
)
print(Enseignant.objects.filter(specialite='Algorithmique').values('nom', 'specialite'))

IntegrityError: UNIQUE constraint failed: planification_enseignant.email

In [6]:
from planification.models import Enseignant, Disponibilite, Session
import datetime

tall = Enseignant.objects.get(nom="Tall")
session = Session.objects.first()

Disponibilite.objects.create(
    enseignant=tall,
    session=session,
    jour=datetime.date(2026, 6, 20),
    heure_debut=datetime.time(8, 0),
    heure_fin=datetime.time(12, 0)
)
print("Disponibilité ajoutée ✅")

MultipleObjectsReturned: get() returned more than one Enseignant -- it returned 2!

In [7]:
from planification.models import Memoire, Enseignant, Disponibilite, Salle, Soutenance, Jury
import datetime

# 1- trier les memoires par ordre de soutenance
memoires=Memoire.objects.all()

   #1.1 definir la rarete des specialite
# --Une specialite est rare si le nombre de profs ayant cette specialite est petit
compteur={}
enseignants=Enseignant.objects.all()

for enseignant in enseignants:
        compteur[enseignant.specialite] = compteur.get(enseignant.specialite, 0) + 1
    # --trie
memoires_tries=sorted(memoires,key= lambda objet_memoire:compteur.get(objet_memoire.specialite,0 )  )

# 2-trouver pour chque memoire , quels sont les profs qui peuvent etre membre du jury
dispo=Disponibilite.objects.all()
salles=Salle.objects.all()
soutenances=Soutenance.objects.all()




for memoire in memoires_tries:
    creneaux={}
    membre_jury=[]
    creneau_choisi=None
    candidat_juris=[enseignant for enseignant in enseignants if enseignant.specialite == memoire.specialite]
    
    # 3- parmis ces profs quels sont ceux qui sont declarer dispo au meme moment et qui n'ont pas ete assigne a un jury
    for disp in dispo:
        cle=(disp.jour,disp.heure_debut,disp.heure_fin)
        jour, heure_debut, heure_fin = cle
        soutenances_au_meme_creneau = Soutenance.objects.filter(jour=jour, heure_debut=heure_debut)
        profs_occupes = Jury.objects.filter(soutenance__in=soutenances_au_meme_creneau)

        if cle in creneaux:
            creneaux[cle].extend( [candidat for candidat in candidat_juris if  ( disp.enseignant_id == candidat.id and disp.enseignant_id not in profs_occupes.values_list('enseignant_id', flat=True)  ) ])
            
        else:
            creneaux[cle] =  [candidat for candidat in candidat_juris if ( disp.enseignant_id == candidat.id and disp.enseignant_id not in profs_occupes.values_list('enseignant_id', flat=True))]
        
            
            # 4- S'ils valent 03 , les choisir comme membre du jury 
    for  cle ,liste_juris_probable in creneaux.items():
        if len( liste_juris_probable )>=3:
            membre_jury =liste_juris_probable[:3]
            creneau_choisi = cle
            break
            
    if not creneau_choisi: # Si on trouve pas 03 profs de la meme specialite que la memoire , prendre le nb de prof de la mm specialite et completer avec n'importe quel autre prof
        for  cle ,liste_juris_probable in creneaux.items():
            if 1<= len( liste_juris_probable )<3:
                membre_jury =liste_juris_probable
                creneau_choisi = cle

                break
        if not creneau_choisi:
            continue  # pas de créneau valide -> on passe au mémoire suivant
        
        jour, heure_debut, heure_fin = creneau_choisi
        
        dispo_creneau= Disponibilite.objects.filter(jour=jour, heure_debut=heure_debut, heure_fin=heure_fin )  
        soutenances_au_meme_creneau = Soutenance.objects.filter(jour=jour, heure_debut=heure_debut,heure_fin=heure_fin)
        profs_occupes = Jury.objects.filter(soutenance__in=soutenances_au_meme_creneau)
        
        complement= [disp.enseignant for disp in dispo_creneau if disp.enseignant_id not in profs_occupes.values_list('enseignant_id', flat=True)  ]
        complement = [c for c in complement if c not in membre_jury]
        nombre_manquant = 3 - len(membre_jury)
        membre_jury.extend( [ complete for complete in complement[:nombre_manquant]] )
        
# 5- chercher une salle dispo au creneau choisi
    jour, heure_debut, heure_fin = creneau_choisi
    salle_choisie = None
    for salle in salles:
        occupation = Soutenance.objects.filter(
        jour=jour,
        heure_debut=heure_debut,
        heure_fin=heure_fin,
        salle_id=salle.id
            )
        if  not occupation.exists():
            salle_choisie = salle
            break
    if not salle_choisie:
            continue  # pas de salle dispo → on passe au mémoire suivant
        
 # 6- programmer la soutenance
        
    if len(membre_jury)==3 and salle_choisie : 
            nouvelle_soutenance= Soutenance.objects.create(
            jour=jour,
            heure_debut=heure_debut,
            heure_fin=heure_fin,
            salle_id=salle_choisie.id,
            memoire_id=memoire.id,
            statut='planifiee'
            )
        

# 7-Enregistrons le jury dans la table jury
            for prof in membre_jury:
                Jury.objects.create(
                soutenance_id = nouvelle_soutenance.id,
                enseignant_id =prof.id
                )


def replanifier(session_id):
    # Recuperer les soutenances rejetees
    soutenances_rejetees = Soutenance.objects.filter(
        statut='rejetee',
        memoire__session_id=session_id
    )
 
    # Recuperer les memoires associes avant suppression
    memoires_a_replanifier = [s.memoire for s in soutenances_rejetees]
 
    # Supprimer les jurys et soutenances rejetes
    Jury.objects.filter(soutenance__in=soutenances_rejetees).delete()
    soutenances_rejetees.delete()
 
    # Relancer l'algo uniquement pour ces memoires
    generer_planning(session_id, memoires_cibles=memoires_a_replanifier)
 




In [8]:
for s in Soutenance.objects.all():
    print(f"Mémoire: {s.memoire.titre} | Salle: {s.salle.nom} | Jour: {s.jour} | Heure: {s.heure_debut}")

Mémoire: Architecture microservices | Salle: Salle A | Jour: 2026-06-21 | Heure: 14:00:00
Mémoire: Systemes de recommandation | Salle: Salle B | Jour: 2026-06-21 | Heure: 08:00:00
Mémoire: Methodes agiles en entreprise | Salle: Salle B | Jour: 2026-06-20 | Heure: 08:00:00
Mémoire: Complexite et optimisation | Salle: Salle A | Jour: 2026-06-20 | Heure: 08:00:00
Mémoire: Protocoles de routage avances | Salle: Salle A | Jour: 2026-06-21 | Heure: 08:00:00
Mémoire: Gestion des stocks | Salle: Salle A | Jour: 2026-06-20 | Heure: 08:00:00
Mémoire: Methodes agiles en entreprise | Salle: Salle A | Jour: 2026-06-20 | Heure: 14:00:00
Mémoire: Architecture microservices | Salle: Salle C | Jour: 2026-06-21 | Heure: 08:00:00
Mémoire: Complexite et optimisation | Salle: Salle B | Jour: 2026-06-21 | Heure: 14:00:00
Mémoire: Algorithme de tri | Salle: Salle B | Jour: 2026-06-20 | Heure: 14:00:00


In [9]:
print("Soutenances:", Soutenance.objects.count())
print("Jurys:", Jury.objects.count())

Soutenances: 10
Jurys: 30


In [8]:
print("Mémoires:", Memoire.objects.count())
print("Enseignants:", Enseignant.objects.count())
print("Disponibilités:", Disponibilite.objects.count())

Mémoires: 2
Enseignants: 4
Disponibilités: 20


In [12]:
for memoire in memoires_tries:
    print(f"\nMémoire: {memoire.titre} | Spécialité: {memoire.specialite}")
    candidat_juris = [e for e in enseignants if e.specialite == memoire.specialite]
    print(f"Candidats jury: {[e.nom for e in candidat_juris]}")


Mémoire: Gestion des stocks | Spécialité: Base de données
Candidats jury: ['Kindo']

Mémoire: Algorithme de tri | Spécialité: Algorithmique
Candidats jury: ['Compaore', 'Zongo']


In [19]:
for j in Jury.objects.all():
    print(f"Soutenance: {j.soutenance.memoire.titre} | Prof: {j.enseignant.nom}")

In [18]:
Soutenance.objects.all().delete()
Jury.objects.all().delete()
print("Nettoyé ✅")

Nettoyé ✅


In [ ]:
for j in Jury.objects.all():
    print(f"Soutenance: {j.soutenance.memoire.titre} | Prof: {j.enseignant.nom}")

In [ ]:
print(Enseignant.objects.filter(specialite='Algorithmique').values('nom', 'specialite'))

In [ ]:
from planification.models import Memoire, Enseignant
m = Memoire.objects.get(titre="Algorithme de tri")
print(m.specialite)
print(Enseignant.objects.filter(specialite=m.specialite).count())

In [ ]:
for s in Soutenance.objects.all():
    membres = Jury.objects.filter(soutenance_id=s.id)
    print(f"Soutenance: {s.memoire.titre} | Nombre de jurés: {membres.count()}")

In [ ]:
for s in Soutenance.objects.all():
    membres = Jury.objects.filter(soutenance_id=s.id)
    print(f"Mémoire: {s.memoire.titre} | Jurés: {membres.count()}")

In [10]:
from planification.models import Memoire, Enseignant, Disponibilite, Salle, Soutenance, Jury
import datetime


def generer_planning(session_id, memoires_cibles=None):

    # 1- Recuperer les donnees de la session
    if memoires_cibles is None:
        memoires = Memoire.objects.filter(session_id=session_id)
    else:
        memoires = memoires_cibles

    enseignants = Enseignant.objects.all()
    dispo = Disponibilite.objects.filter(session_id=session_id)
    salles = Salle.objects.all()

    # 1.1 definir la rarete des specialites
    compteur = {}

    for enseignant in enseignants:
        compteur[enseignant.specialite] = compteur.get(enseignant.specialite, 0) + 1

    # Trier les memoires
    memoires_tries = sorted(
        memoires,
        key=lambda objet_memoire: compteur.get(objet_memoire.specialite, 0)
    )

    # 2- Trouver pour chaque memoire les profs possibles
    for memoire in memoires_tries:

        creneaux = {}
        membre_jury = []
        creneau_choisi = None

        candidat_juris = [
            enseignant
            for enseignant in enseignants
            if enseignant.specialite == memoire.specialite
        ]

        # 3- Chercher les profs disponibles et non occupes
        for disp in dispo:

            cle = (disp.jour, disp.heure_debut, disp.heure_fin)

            jour, heure_debut, heure_fin = cle

            soutenances_au_meme_creneau = Soutenance.objects.filter(
                jour=jour,
                heure_debut=heure_debut,
                heure_fin=heure_fin
            )

            profs_occupes = Jury.objects.filter(
                soutenance__in=soutenances_au_meme_creneau
            )

            liste_profs = [
                candidat
                for candidat in candidat_juris
                if (
                    disp.enseignant_id == candidat.id
                    and disp.enseignant_id not in profs_occupes.values_list('enseignant_id', flat=True)
                )
            ]

            if cle in creneaux:
                creneaux[cle].extend(liste_profs)
            else:
                creneaux[cle] = liste_profs

        # 4- Si au moins 3 profs de la specialite
        for cle, liste_juris_probable in creneaux.items():

            if len(liste_juris_probable) >= 3:

                membre_jury = liste_juris_probable[:3]
                creneau_choisi = cle
                break

        # Sinon completer avec d'autres profs
        if not creneau_choisi:

            for cle, liste_juris_probable in creneaux.items():

                if 1 <= len(liste_juris_probable) < 3:

                    membre_jury = liste_juris_probable
                    creneau_choisi = cle
                    break

            # Aucun creneau valide
            if not creneau_choisi:
                continue

            jour, heure_debut, heure_fin = creneau_choisi

            dispo_creneau = Disponibilite.objects.filter(
                jour=jour,
                heure_debut=heure_debut,
                heure_fin=heure_fin
            )

            soutenances_au_meme_creneau = Soutenance.objects.filter(
                jour=jour,
                heure_debut=heure_debut,
                heure_fin=heure_fin
            )

            profs_occupes = Jury.objects.filter(
                soutenance__in=soutenances_au_meme_creneau
            )

            complement = [
                disp.enseignant
                for disp in dispo_creneau
                if disp.enseignant_id not in profs_occupes.values_list('enseignant_id', flat=True)
            ]

            complement = [
                c for c in complement
                if c not in membre_jury
            ]

            nombre_manquant = 3 - len(membre_jury)

            membre_jury.extend(
                complement[:nombre_manquant]
            )

        # 5- Chercher une salle disponible
        jour, heure_debut, heure_fin = creneau_choisi

        salle_choisie = None

        for salle in salles:

            occupation = Soutenance.objects.filter(
                jour=jour,
                heure_debut=heure_debut,
                heure_fin=heure_fin,
                salle_id=salle.id
            )

            if not occupation.exists():

                salle_choisie = salle
                break

        # Pas de salle disponible
        if not salle_choisie:
            continue

        # 6- Programmer la soutenance
        if len(membre_jury) == 3 and salle_choisie:

            nouvelle_soutenance = Soutenance.objects.create(
                jour=jour,
                heure_debut=heure_debut,
                heure_fin=heure_fin,
                salle_id=salle_choisie.id,
                memoire_id=memoire.id,
                statut='planifiee'
            )

            # 7- Enregistrer le jury
            for prof in membre_jury:

                Jury.objects.create(
                    soutenance_id=nouvelle_soutenance.id,
                    enseignant_id=prof.id
                )


def replanifier(session_id):

    # Recuperer les soutenances rejetees
    soutenances_rejetees = Soutenance.objects.filter(
        statut='rejetee',
        memoire__session_id=session_id
    )

    # Recuperer les memoires associes
    memoires_a_replanifier = [
        soutenance.memoire
        for soutenance in soutenances_rejetees
    ]

    # Supprimer les jurys
    Jury.objects.filter(
        soutenance__in=soutenances_rejetees
    ).delete()

    # Supprimer les soutenances
    soutenances_rejetees.delete()

    # Relancer l'algo
    generer_planning(
        session_id,
        memoires_cibles=memoires_a_replanifier
    )



In [13]:
# Garder seulement la première session
Session.objects.exclude(id=1).delete()
print("Sessions nettoyées ✅")

Sessions nettoyées ✅


In [11]:
from django.contrib.auth.models import User
from planification.models import Enseignant

# Créer un compte utilisateur pour Kindo
user_kindo = User.objects.create_user(
    username='kindo',
    password='kindo1234'
)

# Lier ce compte à l'enseignant Kindo
kindo = Enseignant.objects.get(nom='Kindo')
kindo.user = user_kindo
kindo.save()

print("Compte Kindo créé ✅")

IntegrityError: UNIQUE constraint failed: auth_user.username

In [16]:
# Garder seulement la première session
Session.objects.exclude(id=Session.objects.first().id).delete()
print("✅ Nettoyé")

✅ Nettoyé
